# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanaahmedradwan123-commits/flyrank-internship-w1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from huggingface_hub import login
import os

# Load token from secret
token = os.getenv('HF_TOKEN')

# Authenticate
login(token=token)

print("✓ Authenticated with Hugging Face")

✓ Authenticated with Hugging Face


In [2]:
from datasets import load_dataset

configs = ['dim_clients', 'dim_content', 'fact_content_daily_performance', 'fact_content_query_90d']

for config in configs:
    try:
        ds = load_dataset("FlyRank/internship-warehouse", config)
        print(f"\n{'='*70}")
        print(f"CONFIG: {config}")
        print(f"{'='*70}")
        print(f"Columns: {ds['train'].column_names}")
        print(f"Row count: {len(ds['train'])}")
        print(f"\nFirst row:\n{ds['train'][0]}\n")
    except Exception as e:
        print(f"Error loading {config}: {e}")


CONFIG: dim_clients
Columns: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']
Row count: 104

First row:
{'client_hash_id': 'client_04660893ae39614a', 'is_active': True, 'has_gsc_access': True, 'has_ga4_access': True, 'access_profile': 'gsc_and_ga4', 'client_created_date': datetime.date(2026, 4, 15), 'client_updated_date': datetime.date(2026, 6, 27), 'gsc_data_start': None, 'ga4_data_start': datetime.date(2026, 5, 22)}


CONFIG: dim_content
Columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_e

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]


CONFIG: fact_content_daily_performance
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
Row count: 78835655

First row:
{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pageviews': 0, 'ga4_sessions': 0, 'ga4_use

## 1. Unit of analysis + time window

One row = one (client_hash_id, query_hash_id, content_hash_id) triple,
representing how a piece of content performed for a specific query within a 90-day
rolling window. The window_start and window_end define when the observation began
and ended.

Time window: 90 days (rolling). We'll develop on March 2026 data
(filter to window_start in 2026-03) and treat June 2026 as a sealed test month.

Label: clicks_90d (count of clicks in the 90-day window).
This is what we'll predict.

Exclusions:
- Rows where impressions_90d < 3 (too sparse to trust).
- Rows where window_end is in June 2026 (sealed test month, no development).

## 2. Fields: feature / label / context / excluded

FEATURES (knowable at decision time):
- query_char_count: number of characters in query
- query_token_count: number of tokens in query
- impressions_prev30: impressions in the 30 days before last30
- clicks_prev30: clicks in the 30 days before last30
- avg_position_prev30: average ranking position in prev30
- content_total_impressions_90d: total impressions for this content across all queries
- content_visible_query_count: how many queries this content appeared for
- rare_query_count: number of rare (low-volume) queries this content serves
- rare_impressions_share: % of impressions from rare queries
- anonymized_impressions_share: % of impressions that were anonymized

LABEL (what we predict):
- clicks_90d: count of clicks in the 90-day window

CONTEXT (needed for grouping, not modeling):
- client_hash_id: which client
- query_hash_id: which query
- content_hash_id: which content
- window_start, window_end: observation window dates

EXCLUDED (too leaky or too sparse):
- impressions_last30: this leaks future clicks_last30 → skip
- clicks_last30: this IS future clicks_90d → skip
- avg_position_last30: this is also future information → skip
- impressions_90d: this is the denominator of our label, not a predictor

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
import pandas as pd
from datasets import load_dataset
from datetime import datetime

# Load the dataset
ds = load_dataset("FlyRank/internship-warehouse", 'fact_content_query_90d')
df = ds['train'].to_pandas()

# Convert window_start to datetime
df['window_start'] = pd.to_datetime(df['window_start'])
df['window_month'] = df['window_start'].dt.to_period('M')

print("=" * 80)
print("VERIFICATION QUERY 1: Grain Check")
print("=" * 80)
# Query 1: Verify one row = one (client, query, content) triple
april_df = df[df['window_month'] == '2026-04']
grain_check = april_df.groupby(
    ['client_hash_id', 'query_hash_id', 'content_hash_id']
).size().reset_index(name='count')

print(f"Total (client, query, content) triples in April 2026: {len(grain_check)}")
print(f"Max rows per triple: {grain_check['count'].max()}")
print(f"Min rows per triple: {grain_check['count'].min()}")
print(f"✓ Grain verified: Each triple appears exactly once" if grain_check['count'].max() == 1 else "✗ WARNING: Duplicates found!")

print("\n" + "=" * 80)
print("VERIFICATION QUERY 2: Row count and date span")
print("=" * 80)
# Query 2: Count rows in April 2026 and show date range
print(f"Total rows in April 2026: {len(april_df)}")
print(f"Date span (window_start): {april_df['window_start'].min()} to {april_df['window_start'].max()}")
print(f"Date span (window_end): {pd.to_datetime(april_df['window_end']).min()} to {pd.to_datetime(april_df['window_end']).max()}")

print("\n" + "=" * 80)
print("VERIFICATION QUERY 3: Availability check (IS TRUE)")
print("=" * 80)
# Query 3: Availability — how many rows pass the exclusion filter?
# Exclusion: impressions_90d >= 3
april_df['is_usable'] = april_df['impressions_90d'] >= 3

print(f"Total rows in April 2026: {len(april_df)}")
print(f"Rows passing IS TRUE (impressions_90d >= 3): {april_df['is_usable'].sum()}")
print(f"Availability: {100 * april_df['is_usable'].mean():.1f}%")
print(f"\nRows with impressions_90d < 3 (excluded): {(~april_df['is_usable']).sum()}")

VERIFICATION QUERY 1: Grain Check
Total (client, query, content) triples in April 2026: 2414248
Max rows per triple: 1
Min rows per triple: 1
✓ Grain verified: Each triple appears exactly once

VERIFICATION QUERY 2: Row count and date span
Total rows in April 2026: 2414248
Date span (window_start): 2026-04-02 00:00:00 to 2026-04-02 00:00:00
Date span (window_end): 2026-06-30 00:00:00 to 2026-06-30 00:00:00

VERIFICATION QUERY 3: Availability check (IS TRUE)
Total rows in April 2026: 2414248
Rows passing IS TRUE (impressions_90d >= 3): 2414248
Availability: 100.0%

Rows with impressions_90d < 3 (excluded): 0


In [4]:
import pandas as pd
from datasets import load_dataset

# Load and filter to April 2026
ds = load_dataset("FlyRank/internship-warehouse", 'fact_content_query_90d')
df = ds['train'].to_pandas()
df['window_start'] = pd.to_datetime(df['window_start'])
df['window_month'] = df['window_start'].dt.to_period('M')
april_df = df[df['window_month'] == '2026-04'].copy()

# Build five-feature frame
features_df = april_df[[
    'client_hash_id', 'query_hash_id', 'content_hash_id',
    'query_token_count',           # Feature 1
    'impressions_prev30',          # Feature 2
    'avg_position_prev30',         # Feature 3
    'content_total_impressions_90d',  # Feature 4
    'content_visible_query_count'  # Feature 5
]].copy()

# Add label
features_df['clicks_90d'] = april_df['clicks_90d'].values

print("=" * 80)
print("FIVE-FEATURE FRAME (April 2026)")
print("=" * 80)
print(f"Shape: {features_df.shape}")
print(f"\nFirst 5 rows:\n{features_df.head()}")

print("\n" + "=" * 80)
print("FEATURE DESCRIPTIONS & AVAILABILITY JUSTIFICATIONS")
print("=" * 80)

features_info = [
    ("query_token_count", "Knowable at decision time because: The query text is known the moment a search is made. We can tokenize it immediately."),
    ("impressions_prev30", "Knowable at decision time because: This is historical data from the previous 30 days, fully observed before we rank this query today."),
    ("avg_position_prev30", "Knowable at decision time because: This is the ranking position from 30 days ago, historical context about how well this content ranked for this query before."),
    ("content_total_impressions_90d", "Knowable at decision time because: This aggregates impressions for the content across all queries in the past 90 days—a measure of content popularity, available before we rank."),
    ("content_visible_query_count", "Knowable at decision time because: This counts how many different queries this content has appeared for—a breadth measure. Fully observable from past data."),
]

for i, (feature, justification) in enumerate(features_info, 1):
    print(f"\nFeature {i}: {feature}")
    print(f"  {justification}")
    print(f"  Nulls: {features_df[feature].isna().sum()} | Mean: {features_df[feature].mean():.2f} | Max: {features_df[feature].max():.2f}")

print("\n" + "=" * 80)
print("LABEL STATISTICS")
print("=" * 80)
print(f"clicks_90d: Mean = {features_df['clicks_90d'].mean():.3f}, Max = {features_df['clicks_90d'].max()}, Nulls = {features_df['clicks_90d'].isna().sum()}")

FIVE-FEATURE FRAME (April 2026)
Shape: (2414248, 9)

First 5 rows:
            client_hash_id           query_hash_id           content_hash_id  \
0  client_08a6a72ff48e62c0  query_58b1b001f839d699  content_447894f2faf0d2bc   
1  client_08a6a72ff48e62c0  query_922b8eca2a24cd34  content_447894f2faf0d2bc   
2  client_08a6a72ff48e62c0  query_9f0c36a6ae2a6a99  content_447894f2faf0d2bc   
3  client_08a6a72ff48e62c0  query_a032820b5467e996  content_447894f2faf0d2bc   
4  client_08a6a72ff48e62c0  query_ba1a2f131961c5da  content_447894f2faf0d2bc   

   query_token_count  impressions_prev30  avg_position_prev30  \
0                  3                  11            10.818182   
1                  7                   1            11.000000   
2                  2                   5            22.000000   
3                  4                   1             0.000000   
4                  3                   0                  NaN   

   content_total_impressions_90d  content_visible_query_count

SECTION 5: THE LEAKAGE TRAP (PERFORMED & REMOVED)

We deliberately added impressions_last30 as a feature and watched the R² score
jump artificially. Here's why it's a trap:

- impressions_last30 is derived from the SAME 90-day observation window as
  our label (clicks_90d).
- At DECISION TIME (when we rank today), we don't know yet how many impressions
  will happen in the last 30 days of a future 90-day window.
- In production, this feature doesn't exist.
- The model overfits to this leak and will fail on real data.

We removed it and kept the honest baseline. Real data contracts must be airtight
about what's knowable when.

In [5]:
import pandas as pd
from datasets import load_dataset
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Reload and prepare data
ds = load_dataset("FlyRank/internship-warehouse", 'fact_content_query_90d')
df = ds['train'].to_pandas()
df['window_start'] = pd.to_datetime(df['window_start'])
df['window_month'] = df['window_start'].dt.to_period('M')
april_df = df[df['window_month'] == '2026-04'].copy()

# Build honest feature frame (the 5 features we approved)
honest_features = [
    'query_token_count',
    'impressions_prev30',
    'avg_position_prev30',
    'content_total_impressions_90d',
    'content_visible_query_count'
]

# Prepare data
X_honest = april_df[honest_features].fillna(0).values
y = april_df['clicks_90d'].values

# Train/test split (80/20)
n = len(X_honest)
split = int(0.8 * n)
X_train, X_test = X_honest[:split], X_honest[split:]
y_train, y_test = y[:split], y[split:]

print("=" * 80)
print("BASELINE MODEL: Honest Features Only")
print("=" * 80)

model_honest = GradientBoostingRegressor(n_estimators=50, random_state=42)
model_honest.fit(X_train, y_train)
y_pred_honest = model_honest.predict(X_test)

r2_honest = r2_score(y_test, y_pred_honest)
rmse_honest = np.sqrt(mean_squared_error(y_test, y_pred_honest))

print(f"R² Score (honest): {r2_honest:.4f}")
print(f"RMSE (honest): {rmse_honest:.4f}")

print("\n" + "=" * 80)
print("LEAKY MODEL: Added label-derived feature (impressions_last30)")
print("=" * 80)
print("⚠️  TRAP: impressions_last30 is part of the 90-day window!")
print("   It's derived from the SAME observation window as clicks_90d.")
print("   This creates leakage: future information predicting itself.\n")

# Add the leaky feature
X_leaky = april_df[honest_features + ['impressions_last30']].fillna(0).values
X_train_leaky, X_test_leaky = X_leaky[:split], X_leaky[split:]

model_leaky = GradientBoostingRegressor(n_estimators=50, random_state=42)
model_leaky.fit(X_train_leaky, y_train)
y_pred_leaky = model_leaky.predict(X_test_leaky)

r2_leaky = r2_score(y_test, y_pred_leaky)
rmse_leaky = np.sqrt(mean_squared_error(y_test, y_pred_leaky))

print(f"R² Score (leaky): {r2_leaky:.4f}")
print(f"RMSE (leaky): {rmse_leaky:.4f}")

print("\n" + "=" * 80)
print("LEAKAGE IMPACT")
print("=" * 80)
r2_jump = r2_leaky - r2_honest
rmse_drop = rmse_honest - rmse_leaky
print(f"R² jumped by: {r2_jump:+.4f} ({100 * r2_jump / abs(r2_honest):.1f}%)")
print(f"RMSE dropped by: {rmse_drop:+.4f} ({100 * rmse_drop / rmse_honest:.1f}%)")
print(f"\n⚠️  The leaky feature gives FALSE confidence.")
print(f"    In production, impressions_last30 won't exist at decision time.")
print(f"    This model will crash in the real world.")

print("\n" + "=" * 80)
print("CONCLUSION: Keep the honest model")
print("=" * 80)
print(f"Honest R²: {r2_honest:.4f}")
print(f"Honest RMSE: {rmse_honest:.4f}")
print(f"\nThis is the real score. The leaky score is a mirage.")

BASELINE MODEL: Honest Features Only
R² Score (honest): 0.2194
RMSE (honest): 1.4457

LEAKY MODEL: Added label-derived feature (impressions_last30)
⚠️  TRAP: impressions_last30 is part of the 90-day window!
   It's derived from the SAME observation window as clicks_90d.
   This creates leakage: future information predicting itself.

R² Score (leaky): 0.3537
RMSE (leaky): 1.3154

LEAKAGE IMPACT
R² jumped by: +0.1343 (61.2%)
RMSE dropped by: +0.1302 (9.0%)

⚠️  The leaky feature gives FALSE confidence.
    In production, impressions_last30 won't exist at decision time.
    This model will crash in the real world.

CONCLUSION: Keep the honest model
Honest R²: 0.2194
Honest RMSE: 1.4457

This is the real score. The leaky score is a mirage.


## 4. Data limits

## What This Data Can Never Tell Us (One Named Limitation)

**Window overlap and label censoring:**

Every row in fact_content_query_90d represents a rolling 90-day window.
But the observation period (April 2 → June 30, 2026) is only ~90 days long.
This means:

1. April rows have 90 days of outcome history (full window closure).
2. June rows have only ~30 days of outcome history (window closes at June 30).
3. This creates a **label bias**: June clicks are artificially suppressed
   because the window hasn't finished accruing.

We cannot predict what June's true 90-day clicks will be. We're predicting
a truncated, censored version. Any model trained here will underestimate
click counts for queries that occur late in the observation window.

**Implication:** Models trained on this data will not transfer to future months
without retraining on data with full window closure.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.